In [44]:
import os
import numpy as np
import librosa
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import f_classif, chi2
from sklearn.preprocessing import KBinsDiscretizer

In [18]:
preprocessed_dir = "data/preprocessed"
classes = ["copd", "healthy"]

features_dir = "data/features"
os.makedirs(features_dir, exist_ok=True)

In [22]:
def extract_features(y, sr=16000, n_mfcc=13):
    
    features = []
    
    #MFCCs (mean + std over frames)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    features.extend(np.mean(mfccs, axis=1))
    features.extend(np.std(mfccs, axis=1))
    
    #Spectral centroid
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    features.append(np.mean(spec_cent))
    features.append(np.std(spec_cent))
    
    #Spectral bandwidth
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    features.append(np.mean(spec_bw))
    features.append(np.std(spec_bw))
    
    #Zero crossing rate
    zcr = librosa.feature.zero_crossing_rate(y)
    features.append(np.mean(zcr))
    features.append(np.std(zcr))
    '''
    #Chroma features
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features.extend(np.mean(chroma, axis=1))
    features.extend(np.std(chroma, axis=1))

    #Root mean square energy
    rms = librosa.feature.rms(y=y)
    features.append(np.mean(rms))
    features.append(np.std(rms))

    #Spectral roll-off
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    features.append(np.mean(rolloff))
    features.append(np.std(rolloff))

    #Tempo 
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    features.append(tempo)

    #Delta MFCCs
    mfcc_delta = librosa.feature.delta(mfccs)
    features.extend(np.mean(mfcc_delta, axis=1))
    features.extend(np.std(mfcc_delta, axis=1))

    #MFCC acceleration
    mfcc_delta2 = librosa.feature.delta(mfccs, order=2)
    features.extend(np.mean(mfcc_delta2, axis=1))
    features.extend(np.std(mfcc_delta2, axis=1))

    #Log mel spectogram features
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr)
    log_mel_spec = librosa.power_to_db(mel_spec)
    features.extend(np.mean(log_mel_spec, axis=1))
    features.extend(np.std(log_mel_spec, axis=1))'''
    
    return np.array(features)

In [25]:
X = []
y_labels = []

for cls in classes:
    cls_dir = os.path.join(preprocessed_dir, cls)
    for fname in os.listdir(cls_dir):
        if not fname.endswith(".npy"):
            continue

        #Load preprocessed numeric array
        y_wave = np.load(os.path.join(cls_dir, fname))

        #Extract features
        feat = extract_features(y_wave, sr=16000)
        X.append(feat)
        y_labels.append(cls)

#Convert to arrays
X = np.array(X)
y_labels = np.array(y_labels)

print("Feature matrix shape:", X.shape)
print("Labels shape:", y_labels.shape)


Feature matrix shape: (1605, 32)
Labels shape: (1605,)


In [37]:
feature_names = []
n_mfcc = 13
for i in range(n_mfcc):
    feature_names.append(f"mfcc_{i+1}_mean")
for i in range(n_mfcc):
    feature_names.append(f"mfcc_{i+1}_std")

feature_names.extend([
    "spec_centroid_mean",
    "spec_centroid_std",
    "spec_bandwidth_mean",
    "spec_bandwidth_std",
    "zcr_mean",
    "zcr_std"
])

In [38]:
#Convert to dataframe for feature analysis
df = pd.DataFrame(X, columns=feature_names)
df['label'] = y_labels

In [64]:
#Save features for ML
np.save(os.path.join(features_dir, "X.npy"), X)
np.save(os.path.join(features_dir, "y.npy"), y_labels)

with open(os.path.join(features_dir, "dataset.pkl"), "wb") as f:
    pickle.dump(
        {"X": X, "y": y_labels, "feature_names": feature_names},
        f
    )